# Fabric Notebook: HR Map (wf_m_poc_xml_hr)
**Source:** XML (Hierarchical) → **Target:** CSV (Flattened)

**Created:** 2026-06-19

This notebook implements the PowerCenter HR hierarchical mapping for Fabric - load, flatten and transform hierarchical HR data from XML to CSV.

## SECTION 1: IMPORTS & CONFIGURATION

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, 
    when, 
    trim, 
    upper, 
    cast,
    explode,
    concat_ws,
    current_timestamp,
    count,
    avg,
    round,
    max,
    min
)
from pyspark.sql.types import *
import os
from datetime import datetime

# Configuration
LAKEHOUSE_PATH = "/lakehouse/default/Files"
SOURCE_FILE = "hr.xml"
TARGET_LOCATION = "hr_poc_target"

print(f"📍 Map: HR Source (Hierarchical) → Target (Flattened)")
print(f"⏰ Execution Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔹 Lakehouse Path: {LAKEHOUSE_PATH}")
print("=" * 80)

## SECTION 2: SOURCE EXTRACTION (Hierarchical XML Reader)

In [ ]:
print("\n📥 [STEP 1] Reading Hierarchical XML Source")
print("-" * 80)

try:
    # Read XML with hierarchical structure
    df_source = spark.read \
        .format("xml") \
        .option("rowTag", "Department") \
        .option("inferSchema", "true") \
        .load(f"{LAKEHOUSE_PATH}/{SOURCE_FILE}")
    
    print(f"✓ Hierarchical XML Source loaded successfully")
    print(f"  Root structure: Department")
    print(f"  Nested elements: Employees/Employee")
    
    print("\n📋 Source Schema:")
    df_source.printSchema()
    
except Exception as e:
    print(f"❌ Error reading XML source: {str(e)}")
    raise

## SECTION 3: FLATTEN HIERARCHICAL DATA

In [ ]:
print("\n🔄 [STEP 2] Flattening Hierarchical Structure")
print("-" * 80)

try:
    # Explode the Employees array to flatten
    df_flattened = df_source.select(
        col("DEPT_ID"),
        col("DEPT_NAME"),
        explode(col("Employees.Employee")).alias("employee")
    ).select(
        col("DEPT_ID"),
        col("DEPT_NAME"),
        col("employee.EMP_ID"),
        col("employee.FIRST_NAME"),
        col("employee.LAST_NAME"),
        col("employee.SALARY")
    )
    
    print(f"✓ Hierarchical structure flattened")
    print(f"  Total rows after flattening: {df_flattened.count()}")
    
    df_flattened.show(10)
    
except Exception as e:
    print(f"❌ Error flattening hierarchical data: {str(e)}")
    raise

## SECTION 4: DATA TRANSFORMATION (Map Logic)

In [ ]:
print("\n🔄 [STEP 3] Applying Transformations")
print("-" * 80)

try:
    # Apply map transformations
    df_transformed = df_flattened.select(
        cast(col("DEPT_ID"), ByteType()).alias("DEPT_ID"),
        trim(upper(col("DEPT_NAME"))).alias("DEPT_NAME"),
        cast(col("EMP_ID"), ByteType()).alias("EMP_ID"),
        trim(upper(col("FIRST_NAME"))).alias("FIRST_NAME"),
        trim(upper(col("LAST_NAME"))).alias("LAST_NAME"),
        cast(col("SALARY"), ShortType()).alias("SALARY"),
    )
    
    print(f"✓ Transformations applied")
    print(f"  Rows processed: {df_transformed.count()}")
    print(f"  Data types verified")
    
    df_transformed.show(10)
    
    # Display schema
    print("\n📋 Target Schema:")
    df_transformed.printSchema()
    
except Exception as e:
    print(f"❌ Error applying transformations: {str(e)}")
    raise

## SECTION 5: DATA QUALITY CHECKS

In [ ]:
print("\n✅ [STEP 4] Data Quality Validation")
print("-" * 80)

try:
    # Check for nulls in key fields
    null_check = df_transformed.filter(
        (col("DEPT_ID").isNull()) | 
        (col("EMP_ID").isNull()) | 
        (col("FIRST_NAME").isNull())
    ).count()
    
    print(f"  Null checks (key fields): {null_check} issues found")
    
    # Check for duplicates by composite key
    total_rows = df_transformed.count()
    distinct_rows = df_transformed.select("DEPT_ID", "EMP_ID").distinct().count()
    dup_check = total_rows - distinct_rows
    
    print(f"  Duplicate checks (DEPT_ID + EMP_ID): {dup_check} duplicates found")
    
    # Salary range validation
    salary_check = df_transformed.filter(col("SALARY") < 0).count()
    print(f"  Salary validation (< 0): {salary_check} issues found")
    
    # Department validation
    dept_count = df_transformed.select("DEPT_ID").distinct().count()
    print(f"  Department count: {dept_count} unique departments")
    
    if null_check == 0 and dup_check == 0 and salary_check == 0:
        print(f"✓ All data quality checks PASSED")
    else:
        print(f"⚠️  Some data quality issues detected")
    
except Exception as e:
    print(f"❌ Error during validation: {str(e)}")
    raise

## SECTION 6: HIERARCHICAL RELATIONSHIP VALIDATION

In [ ]:
print("\n🔗 [STEP 5] Validating Hierarchical Relationships")
print("-" * 80)

try:
    # Group by department to validate structure
    dept_summary = df_transformed.groupBy("DEPT_ID", "DEPT_NAME") \
        .agg(
            count("EMP_ID").alias("employee_count"),
            round(avg("SALARY"), 2).alias("avg_salary"),
            max("SALARY").alias("max_salary"),
            min("SALARY").alias("min_salary")
        ).orderBy("DEPT_ID")
    
    print(f"✓ Hierarchical relationships validated")
    print(f"\n📊 Department Summary:")
    dept_summary.show(truncate=False)
    
except Exception as e:
    print(f"❌ Error validating relationships: {str(e)}")
    raise

## SECTION 7: TARGET OUTPUT (CSV Writer)

In [ ]:
print("\n📤 [STEP 7] Writing to Target CSV")
print("-" * 80)

try:
    # Write to CSV
    target_path = f"{LAKEHOUSE_PATH}/{TARGET_LOCATION}"
    
    df_transformed.coalesce(1).write \
        .format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .option("delimiter", ",") \
        .save(target_path)
    
    print(f"✓ Target CSV written successfully")
    print(f"  Location: {target_path}")
    print(f"  Records: {df_transformed.count()}")
    
except Exception as e:
    print(f"❌ Error writing target CSV: {str(e)}")
    raise

## SECTION 8: SUMMARY REPORT

In [ ]:
print("\n📊 [SUMMARY] Map Execution Report")
print("=" * 80)
print(f"Map Name:          wf_m_poc_xml_hr")
print(f"Source Format:     XML (Hierarchical)")
print(f"Target Format:     CSV (Flattened)")
print(f"Source Records:    {df_source.count()}")
print(f"Flattened Records: {df_transformed.count()}")
print(f"Departments:       {df_transformed.select('DEPT_ID').distinct().count()}")
print(f"Execution Status:  ✓ SUCCESS")
print(f"Timestamp:         {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

## SECTION 9: CREATE DELTA TABLES (Optional)

In [ ]:
print("\n🔹 [STEP 9] Creating Delta Tables (Optional)")
print("-" * 80)

try:
    # Drop existing tables if they exist
    spark.sql("DROP TABLE IF EXISTS hr_departments")
    spark.sql("DROP TABLE IF EXISTS hr_employees")
    
    # Create departments table
    df_departments = df_transformed.select("DEPT_ID", "DEPT_NAME").distinct()
    df_departments.write.format("delta").mode("overwrite").saveAsTable("hr_departments")
    print(f"✓ Delta table 'hr_departments' created")
    
    # Create employees table with department FK
    df_transformed.write.format("delta").mode("overwrite").saveAsTable("hr_employees")
    print(f"✓ Delta table 'hr_employees' created")
    
    # Verify
    dept_count = spark.sql("SELECT COUNT(*) as count FROM hr_departments").collect()[0][0]
    emp_count = spark.sql("SELECT COUNT(*) as count FROM hr_employees").collect()[0][0]
    print(f"  hr_departments: {dept_count} rows")
    print(f"  hr_employees: {emp_count} rows")
    
except Exception as e:
    print(f"⚠️  Optional step skipped: {str(e)}")

print("\n✅ Map execution completed successfully!")